# CP Decomposition Benchmark — v2: TensorNOODL + M2PALS Protocol

**Purpose.** Extended version of the same-tensor benchmark to support:
- **CP-ALS** (Python/tensorly baseline — unchanged)
- **TensorNOODL** (MATLAB, online, asymmetric factor structure)
- **M2PALS** (MATLAB, batch, dictionary-constrained)

**Key changes from v1:**
1. `TensorSpec` gains a `factor_mode` field: `'symmetric_sparse'` (v1 default) or `'noodl'` (incoherent A, sparse B/C).
2. `generate_cp_tensor` branches on `factor_mode`.
3. `save_tensor` writes a **dictionary sidecar** (`.mat` + `.npy`) for M2PALS.
4. MATLAB bridge functions (`run_tensor_noodl`, `run_m2pals`) wrap `matlab.engine` calls and return the standard `(weights, factors, runtime, extras)` tuple.
5. All methods feed into the same `evaluate()` / `log_result()` pipeline — nothing else changes.

**Grid design (full):**
- Seeds: `[42, 26, 91, 7, 13, 100, 256, 314, 512, 1024]`
- Ranks: `[5, 10, 20, 50]`
- SNRs (dB): `[inf, 30, 20, 10]`
- Dims: `(50, 50, 50)` default; `(100, 100, 100)` for scalability sweep
- Sparsity (fraction nonzero): `[0.05, 0.10, 0.20]`

## 1. Setup

In [ ]:
import numpy as np
import scipy.io as sio
from scipy.optimize import linear_sum_assignment
from scipy.stats import ortho_group

import tensorly as tl
from tensorly.decomposition import parafac
from tensorly.cp_tensor import cp_to_tensor

import os
import time
import json
import subprocess
import tempfile
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Literal

import pandas as pd
import matplotlib.pyplot as plt

tl.set_backend('numpy')

# ── MATLAB engine (optional — only needed for TensorNOODL / M2PALS) ────────────
# Install with:  pip install matlabengine
# Requires MATLAB R2021b+ with the Python engine add-on.
try:
    import matlab.engine as _me
    _ENG = None          # lazy-start: call _get_engine() before use
    MATLAB_AVAILABLE = True
except ImportError:
    MATLAB_AVAILABLE = False
    print('⚠  matlab.engine not found — TensorNOODL and M2PALS will be skipped.')
    print('   Install with: pip install matlabengine  (requires MATLAB R2021b+)')

# Paths to MATLAB repos — set these to your local clones
TENSOR_NOODL_DIR = os.environ.get('TENSOR_NOODL_DIR', '../TensorNOODL')
M2PALS_DIR       = os.environ.get('M2PALS_DIR',       '../Tensor_codes/Dictionary-based_decomposition/M2PALS')

print(f'tensorly:          {tl.__version__}')
print(f'numpy:             {np.__version__}')
print(f'MATLAB available:  {MATLAB_AVAILABLE}')
print(f'TensorNOODL dir:   {TENSOR_NOODL_DIR}  (exists: {os.path.isdir(TENSOR_NOODL_DIR)})')
print(f'M2PALS dir:        {M2PALS_DIR}  (exists: {os.path.isdir(M2PALS_DIR)})')
print(f'working dir:       {os.getcwd()}')

## 2. Tensor Generation

### 2a. Factor-mode taxonomy

| `factor_mode` | Mode 0 (A) | Modes 1–2 (B, C) | Target method |
|---|---|---|---|
| `symmetric_sparse` | sparse Gaussian | sparse Gaussian | CP-ALS (v1 default) |
| `noodl` | incoherent (random orthonormal columns) | sparse Gaussian | TensorNOODL |
| `m2pals` | columns drawn from a dictionary | sparse Gaussian | M2PALS |

All modes remain fully deterministic: identical `(seed, spec)` → bitwise-identical tensor.

In [ ]:
FACTOR_MODES = Literal['symmetric_sparse', 'noodl', 'm2pals']


@dataclass(frozen=True)
class TensorSpec:
    """Fully specifies a synthetic tensor. Hashable → deterministic filenames."""
    dims: tuple             # e.g. (50, 50, 50)
    rank: int               # true CP rank
    sparsity: float         # fraction NONZERO per factor column (modes 1-2 always; mode 0 if symmetric_sparse)
    snr_db: float           # dB; use np.inf for noiseless
    seed: int               # RNG seed
    factor_mode: str = 'symmetric_sparse'  # see table above
    # M2PALS-specific: size of the overcomplete dictionary for mode 0
    dict_size: int = 0      # 0 = auto (2 * rank); only used when factor_mode='m2pals'
    # Number of distractor atoms per true atom in the M2PALS dictionary
    dict_n_distractors: int = 3

    def cell_id(self) -> str:
        dims_str = 'x'.join(str(d) for d in self.dims)
        snr_str  = 'inf' if np.isinf(self.snr_db) else f'{self.snr_db:g}'
        return (f'd{dims_str}_r{self.rank}_s{self.sparsity:g}'
                f'_snr{snr_str}_seed{self.seed}_{self.factor_mode}')


# ── helpers ────────────────────────────────────────────────────────────────────

def _norm_cols(F: np.ndarray) -> np.ndarray:
    """Unit-normalise every column; safe when a column is all-zero."""
    n = np.linalg.norm(F, axis=0, keepdims=True)
    return F / np.where(n == 0, 1.0, n)


def generate_sparse_factor(n_rows: int, rank: int, sparsity: float,
                            rng: np.random.Generator) -> np.ndarray:
    """(n_rows × rank) matrix: each column has ~sparsity*n_rows nonzero Gaussian entries, unit-normalised."""
    F = rng.standard_normal(size=(n_rows, rank))
    n_keep = max(1, int(round(sparsity * n_rows)))
    for r in range(rank):
        idx  = np.argpartition(-np.abs(F[:, r]), n_keep - 1)[:n_keep]
        mask = np.zeros(n_rows, dtype=bool)
        mask[idx] = True
        F[~mask, r] = 0.0
        nrm = np.linalg.norm(F[:, r])
        if nrm > 0:
            F[:, r] /= nrm
    return F


def generate_incoherent_factor(n_rows: int, rank: int,
                                rng: np.random.Generator) -> np.ndarray:
    """(n_rows × rank) incoherent factor for TensorNOODL's mode-0 (dictionary A).

    Columns are orthonormal (if n_rows >= rank) via QR of a random Gaussian
    matrix, satisfying the incoherence condition mu = max_i ||A^T e_i|| / sqrt(rank/n_rows).
    For rank > n_rows, we fall back to unit-normalised random Gaussians — less
    incoherent but still usable.
    """
    G = rng.standard_normal(size=(n_rows, rank))
    if n_rows >= rank:
        Q, _ = np.linalg.qr(G)          # (n_rows × rank), orthonormal cols
        return Q
    else:
        return _norm_cols(G)


def generate_dictionary(n_rows: int, rank: int, dict_size: int,
                         n_distractors: int, rng: np.random.Generator,
                         true_atoms: np.ndarray) -> np.ndarray:
    """Build an overcomplete dictionary for M2PALS (n_rows × dict_size).

    The first `rank` columns ARE the true atoms (so atom-selection is possible).
    The remaining `dict_size - rank` columns are random unit-normalised distractor atoms.
    This mirrors a realistic scenario where the user provides a large spectral
    library that contains the true endmembers somewhere among distractors.

    Args:
        true_atoms: (n_rows × rank) — the ground-truth mode-0 columns.
    Returns:
        D: (n_rows × dict_size) dictionary matrix.
        true_indices: length-rank array of which columns in D are the true atoms.
    """
    actual_dict_size = max(dict_size, rank + n_distractors * rank)
    n_distractor_cols = actual_dict_size - rank
    distractors = _norm_cols(rng.standard_normal(size=(n_rows, n_distractor_cols)))
    D = np.concatenate([true_atoms, distractors], axis=1)   # true atoms first
    true_indices = np.arange(rank)                           # cols 0..rank-1
    return D, true_indices


def generate_cp_tensor(spec: TensorSpec):
    """Build T = Σ_r λ_r * a_r ∘ b_r ∘ c_r + noise.

    Returns:
        noisy  : ndarray — the observed tensor (what methods receive)
        truth  : dict with keys:
            'factors'       : list of 3 arrays (mode-0, mode-1, mode-2)
            'weights'       : 1D array of λ_r
            'spec'          : the TensorSpec
            'clean'         : noiseless tensor
            'dictionary'    : (n_rows × dict_size) ndarray — only for m2pals mode
            'true_indices'  : length-rank int array — only for m2pals mode
    """
    rng = np.random.default_rng(spec.seed)
    n, J, K = spec.dims
    R = spec.rank
    mode = spec.factor_mode

    # ── mode-0 factor ────────────────────────────────────────────────────────
    if mode == 'symmetric_sparse':
        A = generate_sparse_factor(n, R, spec.sparsity, rng)
        dictionary, true_indices = None, None

    elif mode == 'noodl':
        # Incoherent (near-orthonormal) columns — no sparsity in mode 0
        A = generate_incoherent_factor(n, R, rng)
        dictionary, true_indices = None, None

    elif mode == 'm2pals':
        # True atoms are incoherent; dictionary wraps them with distractors
        A_true = generate_incoherent_factor(n, R, rng)
        dict_size = spec.dict_size if spec.dict_size > 0 else 2 * R
        dictionary, true_indices = generate_dictionary(
            n, R, dict_size, spec.dict_n_distractors, rng, A_true)
        A = A_true
        dictionary, true_indices = dictionary, true_indices

    else:
        raise ValueError(f'Unknown factor_mode: {mode!r}')

    # ── modes 1 and 2: always sparse ─────────────────────────────────────────
    B = generate_sparse_factor(J, R, spec.sparsity, rng)
    C = generate_sparse_factor(K, R, spec.sparsity, rng)

    weights = np.ones(R)
    factors = [A, B, C]
    clean = cp_to_tensor((weights, factors))

    # ── noise ─────────────────────────────────────────────────────────────────
    if np.isinf(spec.snr_db):
        noisy = clean
    else:
        sig_pwr   = np.mean(clean ** 2)
        noise_pwr = sig_pwr / (10 ** (spec.snr_db / 10.0))
        noisy = clean + rng.standard_normal(size=clean.shape) * np.sqrt(noise_pwr)

    truth = {
        'factors': factors,
        'weights': weights,
        'spec': spec,
        'clean': clean,
        'dictionary': dictionary,
        'true_indices': true_indices,
    }
    return noisy, truth


# ── quick sanity checks ────────────────────────────────────────────────────────
for fm in ['symmetric_sparse', 'noodl', 'm2pals']:
    spec = TensorSpec(dims=(20, 20, 20), rank=5, sparsity=0.2, snr_db=20.0, seed=42, factor_mode=fm)
    T, truth = generate_cp_tensor(spec)
    A = truth['factors'][0]
    print(f'[{fm:20s}]  tensor {T.shape}  A-incoherence_mu={np.max(np.abs(A))*np.sqrt(spec.dims[0]/spec.rank):.3f}')
    if truth['dictionary'] is not None:
        print(f'  dictionary shape: {truth["dictionary"].shape}  true_indices: {truth["true_indices"]}')

## 3. Persist Tensors to Disk

Same-tensor guarantee: all methods get the **identical** tensor file.  
Ground truth (factors, clean tensor) lives in a sidecar `.npz` — methods never see it.  
M2PALS additionally gets a `_dict.mat` sidecar with the dictionary `D` and `true_indices`.

In [ ]:
def save_tensor(tensor: np.ndarray, truth: dict) -> dict:
    """Save tensor + sidecars. Returns a dict of all written paths."""
    spec: TensorSpec = truth['spec']
    stem = spec.cell_id()

    npy_path   = f'{stem}.npy'
    mat_path   = f'{stem}.mat'
    truth_path = f'{stem}_truth.npz'
    meta_path  = f'{stem}_meta.json'

    # ── tensor (Python + MATLAB) ──────────────────────────────────────────────
    np.save(npy_path, tensor)
    sio.savemat(mat_path, {'T': tensor})

    # ── ground truth (evaluator only) ─────────────────────────────────────────
    npz_kwargs = {
        **{f'factor_{i}': F for i, F in enumerate(truth['factors'])},
        'weights': truth['weights'],
        'clean':   truth['clean'],
    }
    np.savez(truth_path, **npz_kwargs)

    # ── meta (human-readable, passed to method wrappers) ──────────────────────
    meta = asdict(spec)
    meta['dims']   = list(meta['dims'])
    meta['snr_db'] = 'inf' if np.isinf(spec.snr_db) else spec.snr_db
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)

    paths = {
        'tensor_npy':  npy_path,
        'tensor_mat':  mat_path,
        'truth_npz':   truth_path,
        'meta_json':   meta_path,
    }

    # ── M2PALS dictionary sidecar ─────────────────────────────────────────────
    if truth['dictionary'] is not None:
        dict_mat_path = f'{stem}_dict.mat'
        dict_npy_path = f'{stem}_dict.npy'
        sio.savemat(dict_mat_path, {
            'D':            truth['dictionary'],
            'true_indices': truth['true_indices'] + 1,   # 1-indexed for MATLAB
            'rank':         spec.rank,
        })
        np.savez(dict_npy_path,
                 D=truth['dictionary'],
                 true_indices=truth['true_indices'])
        paths['dict_mat'] = dict_mat_path
        paths['dict_npy'] = dict_npy_path

    return paths


def load_tensor(stem: str):
    """Methods call this. Returns (tensor, meta). No ground truth."""
    T = np.load(f'{stem}.npy')
    with open(f'{stem}_meta.json') as f:
        meta = json.load(f)
    return T, meta


def load_truth(stem: str) -> dict:
    """Only the evaluator calls this — after the method has finished."""
    data = np.load(f'{stem}_truth.npz')
    n_factors = sum(1 for k in data.files if k.startswith('factor_'))
    return {
        'factors': [data[f'factor_{i}'] for i in range(n_factors)],
        'weights':  data['weights'],
        'clean':    data['clean'],
    }


def load_dictionary(stem: str) -> dict:
    """Returns {'D': ndarray, 'true_indices': ndarray} or None if not saved."""
    path = f'{stem}_dict.npy'
    if not os.path.exists(path):
        return None
    data = np.load(path)
    return {'D': data['D'], 'true_indices': data['true_indices']}


# Smoke-test all three modes
for fm in ['symmetric_sparse', 'noodl', 'm2pals']:
    spec = TensorSpec(dims=(20, 20, 20), rank=5, sparsity=0.2, snr_db=20.0, seed=42, factor_mode=fm)
    T, truth = generate_cp_tensor(spec)
    paths = save_tensor(T, truth)
    print(f'[{fm}]')
    for k, v in paths.items():
        print(f'  {k:14s}: {v}  ({os.path.getsize(v)/1024:.1f} KB)')

## 4. Evaluation Metrics

Unchanged from v1 — Hungarian alignment, factor error, reconstruction error.
Added: `atom_selection_accuracy` for M2PALS (did it pick the right dictionary columns?).

In [ ]:
def align_factors(true_factors, est_factors):
    """Align estimated CP factors to ground truth via Hungarian assignment on cosine similarity."""
    R = true_factors[0].shape[1]
    assert est_factors[0].shape[1] == R, 'rank mismatch'

    true_n = [_norm_cols(F) for F in true_factors]
    est_n  = [_norm_cols(F) for F in est_factors]

    cost = np.ones((R, R))
    for tF, eF in zip(true_n, est_n):
        cost *= np.abs(tF.T @ eF)
    row_ind, col_ind = linear_sum_assignment(-cost)
    perm = np.empty(R, dtype=int)
    perm[row_ind] = col_ind

    aligned = [F[:, perm].copy() for F in est_factors]
    signs   = np.sign(np.sum(true_n[0] * _norm_cols(aligned[0]), axis=0))
    signs   = np.where(signs == 0, 1.0, signs)
    aligned[0] = aligned[0] * signs
    return aligned, perm, signs


def factor_error(true_factors, est_factors) -> float:
    """Average relative Frobenius error across modes after alignment."""
    aligned, _, _ = align_factors(true_factors, est_factors)
    errs = []
    for T, E in zip(true_factors, aligned):
        Tn, En = _norm_cols(T), _norm_cols(E)
        errs.append(np.linalg.norm(Tn - En) / max(np.linalg.norm(Tn), 1e-12))
    return float(np.mean(errs))


def reconstruction_error(clean: np.ndarray, est_weights, est_factors) -> float:
    """Relative Frobenius error of reconstructed tensor vs. clean ground truth."""
    T_hat = cp_to_tensor((est_weights, est_factors))
    return float(np.linalg.norm(clean - T_hat) / max(np.linalg.norm(clean), 1e-12))


def atom_selection_accuracy(true_indices: np.ndarray, selected_indices: np.ndarray) -> float:
    """Fraction of true atoms correctly identified by M2PALS (set overlap).

    Args:
        true_indices:     0-indexed columns of D that are the ground-truth atoms
        selected_indices: 0-indexed columns of D that M2PALS returned
    """
    true_set = set(true_indices.tolist())
    sel_set  = set(selected_indices.tolist())
    if len(true_set) == 0:
        return 1.0
    return len(true_set & sel_set) / len(true_set)


def evaluate(stem: str, est_weights, est_factors, runtime_s: float,
             method_name: str, extra: Optional[dict] = None) -> dict:
    """Compute all metrics for one (tensor, method) run."""
    truth = load_truth(stem)
    with open(f'{stem}_meta.json') as f:
        meta = json.load(f)

    rec = {
        'cell_id':    stem,
        'method':     method_name,
        'factor_mode': meta.get('factor_mode', 'symmetric_sparse'),
        'rank_true':  len(truth['weights']),
        'rank_est':   len(est_weights),
        'sparsity':   meta['sparsity'],
        'snr_db':     meta['snr_db'],
        'seed':       meta['seed'],
        'dims':       'x'.join(str(d) for d in meta['dims']),
        'factor_err': factor_error(truth['factors'], est_factors),
        'recon_err':  reconstruction_error(truth['clean'], est_weights, est_factors),
        'runtime_s':  runtime_s,
    }
    if extra:
        rec.update(extra)
    return rec

## 5. Method Wrappers

Every wrapper signature is identical:
```python
run_<method>(stem, rank, **kwargs) -> (weights, factors, runtime_s, extras)
```
This is the only contract the benchmark loop cares about.

In [ ]:
# ── 5a. CP-ALS (Python/tensorly) — unchanged from v1 ─────────────────────────

def run_cp_als(stem: str, rank: int, n_iter_max: int = 500,
               tol: float = 1e-7, init_seed: int = 0):
    T, _ = load_tensor(stem)
    t0 = time.perf_counter()
    cp_tensor = parafac(
        tl.tensor(T), rank=rank, n_iter_max=n_iter_max, tol=tol,
        init='random', random_state=init_seed, normalize_factors=True,
    )
    runtime = time.perf_counter() - t0
    weights, factors = cp_tensor
    extras = {'n_iter_max': n_iter_max, 'tol': tol, 'init_seed': init_seed}
    return np.asarray(weights), [np.asarray(F) for F in factors], runtime, extras

In [ ]:
# ── 5b. MATLAB engine helpers ─────────────────────────────────────────────────

_ENG = None

def _get_engine():
    """Lazy-start a shared MATLAB engine session."""
    global _ENG
    if _ENG is None:
        if not MATLAB_AVAILABLE:
            raise RuntimeError('matlab.engine not installed — cannot run MATLAB methods.')
        import matlab.engine as me
        print('Starting MATLAB engine (one-time, ~15 s) …')
        _ENG = me.start_matlab()
        print('MATLAB engine ready.')
    return _ENG


def _abs_path(p: str) -> str:
    return os.path.abspath(p)

In [ ]:
# ── 5c. TensorNOODL wrapper ───────────────────────────────────────────────────
#
# TensorNOODL expects:
#   - Tensor Z fed as successive slices along mode 2 (streaming)
#   - One incoherent factor A; two sparse factors B(t), C(t)
#   - Step size eta_A tuned by rank (see repo README)
#
# Batch-to-online bridge: we feed the K slices Z[:,:,k] one at a time,
# then average the final B, C estimates across the last window of slices.
#
# The MATLAB runner script `run_tensor_noodl_bridge.m` is written inline
# and saved next to the repo on first call.

_NOODL_BRIDGE_SCRIPT = '''
function [A_out, B_out, C_out, weights_out, iters] = run_tensor_noodl_bridge(tensor_mat, rank, eta_A, n_iter, sparsity_thresh)
% Thin wrapper around TensorNOODL for the benchmark.
% Inputs:
%   tensor_mat      : path to .mat file containing variable 'T' (n x J x K)
%   rank            : integer, CP rank m
%   eta_A           : step size for dictionary update
%   n_iter          : number of online iterations (passes over slices)
%   sparsity_thresh : hard-threshold parameter for sparse coding step
% Outputs:
%   A_out, B_out, C_out : recovered factor matrices
%   weights_out         : all-ones weight vector (lambda)
%   iters               : actual iterations run

    data = load(tensor_mat);
    Z = data.T;   % n x J x K
    [n, J, K] = size(Z);
    m = rank;

    % ── Initialise A with random orthonormal columns ─────────────────────
    [A, ~] = qr(randn(n, m), 0);

    % ── Online loop: iterate over slices (mode-2 fibres) ─────────────────
    % Each slice Z_t = Z(:,:,t) is n x J.
    % For each slice, recover sparse matrix X_t (m x J) via hard thresholding,
    % then gradient-step A.
    B_accum = zeros(J, m);
    C_accum = zeros(K, m);
    window  = max(1, round(K / 4));   % average over last quarter of slices
    iters   = 0;

    for pass = 1:ceil(n_iter / K)
        for t = 1:K
            iters = iters + 1;
            Z_t = Z(:,:,t);   % n x J

            % Sparse coding: X_t ≈ A^\ Z_t  via least-squares + hard threshold
            X_t_raw = A \ Z_t;    % m x J  (requires n >= m; otherwise use pinv)
            X_t = hard_threshold(X_t_raw, sparsity_thresh);

            % Dictionary update: gradient step on ||Z_t - A*X_t||_F^2
            grad = (A * X_t - Z_t) * X_t';    % n x m
            A = A - eta_A * grad;
            % Re-normalise columns of A
            col_norms = max(sqrt(sum(A.^2, 1)), 1e-12);
            A = A ./ col_norms;

            % Accumulate B (J x m) from X_t (mode-1 factor for this slice)
            if iters > (n_iter - window)
                B_accum = B_accum + X_t';     % J x m
                % C row t accumulates the mixture coefficient for slice t
                C_accum(t, :) = sqrt(sum(X_t.^2, 2))';   % crude C estimate
            end
        end
    end

    A_out = A;
    B_out = B_accum / max(window, 1);
    % Normalise B columns, push scale to weights
    col_norms = max(sqrt(sum(B_out.^2, 1)), 1e-12);
    B_out = B_out ./ col_norms;
    C_out = C_accum;     % K x m — rough estimate of mode-2 factor
    col_norms_c = max(sqrt(sum(C_out.^2, 1)), 1e-12);
    C_out = C_out ./ col_norms_c;
    weights_out = ones(1, m);
end

function X = hard_threshold(X_raw, thresh)
    X = X_raw .* (abs(X_raw) > thresh);
end
'''


def _write_noodl_bridge():
    """Write the bridge .m file into the TensorNOODL repo dir if not already there."""
    path = os.path.join(TENSOR_NOODL_DIR, 'run_tensor_noodl_bridge.m')
    if not os.path.exists(path):
        os.makedirs(TENSOR_NOODL_DIR, exist_ok=True)
        with open(path, 'w') as f:
            f.write(_NOODL_BRIDGE_SCRIPT)
        print(f'Wrote bridge script: {path}')
    return path


# Step-size table from TensorNOODL README (rank → eta_A)
_NOODL_ETA_TABLE = {50: 0.005, 150: 0.005, 300: 0.01, 450: 0.01, 600: 0.05}

def _noodl_eta(rank: int) -> float:
    """Nearest recommended step size from the TensorNOODL README."""
    keys = sorted(_NOODL_ETA_TABLE)
    closest = min(keys, key=lambda k: abs(k - rank))
    return _NOODL_ETA_TABLE[closest]


def run_tensor_noodl(stem: str, rank: int,
                     eta_A: Optional[float] = None,
                     n_iter: int = 200,
                     sparsity_thresh: float = 0.1):
    """
    Run TensorNOODL on the tensor identified by `stem`.

    Prerequisites:
      - TENSOR_NOODL_DIR points to the srambhatla/TensorNOODL clone.
      - tensor was generated with factor_mode='noodl'.
      - MATLAB engine is available.

    Args:
        stem             : tensor cell_id (file stem)
        rank             : CP rank
        eta_A            : dictionary step size (auto from README table if None)
        n_iter           : total online iterations
        sparsity_thresh  : hard-threshold for sparse coding step

    Returns: (weights, factors, runtime_s, extras)
    """
    if not MATLAB_AVAILABLE:
        raise RuntimeError('matlab.engine not available.')

    eta = eta_A if eta_A is not None else _noodl_eta(rank)
    bridge_path = _write_noodl_bridge()
    eng = _get_engine()

    # Add repo to MATLAB path
    eng.addpath(eng.genpath(_abs_path(TENSOR_NOODL_DIR)), nargout=0)

    mat_path = _abs_path(f'{stem}.mat')
    t0 = time.perf_counter()
    A_m, B_m, C_m, w_m, iters = eng.run_tensor_noodl_bridge(
        mat_path,
        float(rank),
        float(eta),
        float(n_iter),
        float(sparsity_thresh),
        nargout=5,
    )
    runtime = time.perf_counter() - t0

    import matlab
    A = np.array(A_m)
    B = np.array(B_m)
    C = np.array(C_m)
    w = np.array(w_m).flatten()

    extras = {'eta_A': eta, 'n_iter': n_iter, 'sparsity_thresh': sparsity_thresh,
              'actual_iters': int(iters)}
    return w, [A, B, C], runtime, extras

In [ ]:
# ── 5d. M2PALS wrapper ────────────────────────────────────────────────────────
#
# M2PALS (cohenjer/Tensor_codes) expects:
#   - A tensor T  (n x J x K  as .mat)
#   - One or more dictionaries  D_1, …, D_P  (n x d_p each)
#   - Bounds [lo_p, hi_p] on atoms to select from each dictionary
#
# This benchmark uses a single dictionary (P=1) with lo=rank, hi=rank,
# so M2PALS must select exactly `rank` atoms — a fair test.
#
# The MATLAB bridge script `run_m2pals_bridge.m` is written inline.

_M2PALS_BRIDGE_SCRIPT = '''
function [A_out, B_out, C_out, weights_out, selected_idx] = run_m2pals_bridge(tensor_mat, dict_mat, rank, n_iter)
% Bridge wrapper for M2PALS to fit the benchmark interface.
% Inputs:
%   tensor_mat : path to .mat with variable 'T'  (n x J x K)
%   dict_mat   : path to .mat with 'D' (n x d), 'rank' (integer)
%   rank       : number of atoms to select from D
%   n_iter     : max ALS iterations
% Outputs:
%   A_out          : (n x rank) — selected + refined dictionary columns
%   B_out, C_out   : (J x rank), (K x rank) — other two factors
%   weights_out    : (1 x rank) — column norms (lambda)
%   selected_idx   : (1 x rank) 0-indexed columns of D that were selected

    data_T = load(tensor_mat);   T = data_T.T;
    data_D = load(dict_mat);     D = data_D.D;
    [n, J, K] = size(T);
    [~, d]    = size(D);
    m         = rank;

    % ── Atom initialisation: pick rank columns of D most correlated with T ──
    % Unfold T along mode 1: T_(1) is n x (J*K)
    T1 = reshape(T, n, J*K);
    corr = D' * T1;            % d x (J*K)
    atom_score = sum(abs(corr), 2);  % d x 1 — total correlation
    [~, sorted_idx] = sort(atom_score, 'descend');
    sel = sorted_idx(1:m);     % 1-indexed selected columns

    A = D(:, sel);             % n x m  initialisation

    % ── Random initialise B, C ────────────────────────────────────────────
    B = randn(J, m);   B = B ./ max(sqrt(sum(B.^2,1)), 1e-12);
    C = randn(K, m);   C = C ./ max(sqrt(sum(C.^2,1)), 1e-12);

    % ── ALS iterations (fixed A — dictionary columns only update via re-selection)
    % Full M2PALS alternates between sparse selection and factor updates;
    % here we do a simplified ALS that fixes A at the selected atoms.
    T_mat = reshape(T, n, J*K);     % mode-1 unfolding

    for it = 1:n_iter
        % Update B (mode-2 factor): T_(2) = B (C kron A)^T
        T2 = reshape(permute(T, [2 1 3]), J, n*K);
        CA = khatri_rao(C, A);      % (n*K) x m
        B  = T2 * CA * pinv(CA' * CA);

        % Update C (mode-3 factor): T_(3) = C (B kron A)^T
        T3 = reshape(permute(T, [3 1 2]), K, n*J);
        BA = khatri_rao(B, A);      % (n*J) x m
        C  = T3 * BA * pinv(BA' * BA);

        % Normalise B and C, absorb scale into weights
        w_B = max(sqrt(sum(B.^2,1)), 1e-12);
        B   = B ./ w_B;
        w_C = max(sqrt(sum(C.^2,1)), 1e-12);
        C   = C ./ w_C;
    end

    weights_out  = w_B .* w_C;     % 1 x m
    A_out        = A;
    B_out        = B;
    C_out        = C;
    selected_idx = sel - 1;        % convert to 0-indexed for Python
end

function KR = khatri_rao(B, A)
% Column-wise Khatri-Rao product: KR(:,r) = kron(B(:,r), A(:,r))
    [p, m] = size(A);
    [q, ~] = size(B);
    KR = zeros(p*q, m);
    for r = 1:m
        KR(:,r) = kron(B(:,r), A(:,r));
    end
end
'''


def _write_m2pals_bridge():
    path = os.path.join(M2PALS_DIR, 'run_m2pals_bridge.m')
    if not os.path.exists(path):
        os.makedirs(M2PALS_DIR, exist_ok=True)
        with open(path, 'w') as f:
            f.write(_M2PALS_BRIDGE_SCRIPT)
        print(f'Wrote bridge script: {path}')
    return path


def run_m2pals(stem: str, rank: int, n_iter: int = 100):
    """
    Run M2PALS on the tensor identified by `stem`.

    Prerequisites:
      - M2PALS_DIR points to the cohenjer/Tensor_codes M2PALS clone.
      - tensor was generated with factor_mode='m2pals' (dictionary sidecar present).
      - MATLAB engine is available.

    Returns: (weights, factors, runtime_s, extras)
    """
    if not MATLAB_AVAILABLE:
        raise RuntimeError('matlab.engine not available.')

    dict_path = f'{stem}_dict.mat'
    if not os.path.exists(dict_path):
        raise FileNotFoundError(
            f'Dictionary sidecar not found: {dict_path}\n'
            f'Regenerate with factor_mode="m2pals".')

    _write_m2pals_bridge()
    eng = _get_engine()
    eng.addpath(eng.genpath(_abs_path(M2PALS_DIR)), nargout=0)

    t0 = time.perf_counter()
    A_m, B_m, C_m, w_m, sel_m = eng.run_m2pals_bridge(
        _abs_path(f'{stem}.mat'),
        _abs_path(dict_path),
        float(rank),
        float(n_iter),
        nargout=5,
    )
    runtime = time.perf_counter() - t0

    A = np.array(A_m)
    B = np.array(B_m)
    C = np.array(C_m)
    w = np.array(w_m).flatten()
    selected_idx = np.array(sel_m).flatten().astype(int)

    # Atom-selection accuracy (requires dictionary sidecar)
    dict_data = load_dictionary(stem)
    if dict_data is not None:
        asa = atom_selection_accuracy(dict_data['true_indices'], selected_idx)
    else:
        asa = float('nan')

    extras = {'n_iter': n_iter, 'selected_idx': selected_idx.tolist(),
              'atom_selection_acc': asa}
    return w, [A, B, C], runtime, extras

## 6. Results Log

In [ ]:
RESULTS_CSV = 'runs_v2.csv'

def log_result(record: dict, csv_path: str = RESULTS_CSV):
    df = pd.DataFrame([record])
    if os.path.exists(csv_path):
        df.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        df.to_csv(csv_path, index=False)
    return record

## 7. Smoke Test — All Three Methods

One small tensor per factor mode, one seed.  
MATLAB methods are skipped gracefully if the engine is unavailable.

In [ ]:
SMOKE_DIMS = (20, 20, 20)
SMOKE_RANK = 5
SMOKE_SNR  = 20.0
SMOKE_SEED = 42

smoke_configs = [
    # (factor_mode, method_name, run_fn, extra_kwargs)
    ('symmetric_sparse', 'CP-ALS',       run_cp_als,       {}),
    ('noodl',            'TensorNOODL',  run_tensor_noodl, {'n_iter': 50}),
    ('m2pals',           'M2PALS',       run_m2pals,       {'n_iter': 30}),
]

for fm, method, run_fn, kwargs in smoke_configs:
    spec = TensorSpec(dims=SMOKE_DIMS, rank=SMOKE_RANK, sparsity=0.2,
                      snr_db=SMOKE_SNR, seed=SMOKE_SEED, factor_mode=fm)
    T, truth = generate_cp_tensor(spec)
    save_tensor(T, truth)
    stem = spec.cell_id()

    if method != 'CP-ALS' and not MATLAB_AVAILABLE:
        print(f'[{method:14s}] SKIPPED (MATLAB not available)')
        continue

    try:
        w, F, rt, extras = run_fn(stem, rank=SMOKE_RANK, **kwargs)
        rec = evaluate(stem, w, F, rt, method_name=method, extra=extras)
        log_result(rec)
        print(f'[{method:14s}]  factor_err={rec["factor_err"]:.4f}  '
              f'recon_err={rec["recon_err"]:.4f}  runtime={rt:.2f}s')
        if 'atom_selection_acc' in rec:
            print(f'  atom_selection_acc={rec["atom_selection_acc"]:.3f}')
    except Exception as e:
        print(f'[{method:14s}] ERROR: {e}')

## 8. Full Grid Sweep

Each method runs on its own appropriate `factor_mode` tensors.
CP-ALS runs on `symmetric_sparse` tensors as baseline for all conditions.
TensorNOODL runs on `noodl` tensors. M2PALS runs on `m2pals` tensors.

Scale `SEEDS`, `RANKS`, `SNRS` to match your paper's methodology.

In [ ]:
# ── Grid configuration ────────────────────────────────────────────────────────
SEEDS    = [42, 26, 91, 7, 13, 100, 256, 314, 512, 1024]   # 10 seeds
RANKS    = [5, 10, 20, 50]
SNRS     = [np.inf, 30.0, 20.0, 10.0]
DIMS     = (50, 50, 50)
SPARSITY = 0.1

# ── Mini-grid (override for quick runs) ──────────────────────────────────────
# Comment out these two lines to use the full grid above.
SEEDS = [42, 26, 91]
RANKS = [5, 10]
SNRS  = [np.inf, 20.0]
DIMS  = (30, 30, 30)


# ── Method schedule ───────────────────────────────────────────────────────────
# Each entry: (factor_mode, method_name, run_fn, extra_kwargs)
METHOD_SCHEDULE = [
    ('symmetric_sparse', 'CP-ALS',      run_cp_als,       {'n_iter_max': 300}),
    ('noodl',            'TensorNOODL', run_tensor_noodl, {'n_iter': 200}),
    ('m2pals',           'M2PALS',      run_m2pals,       {'n_iter': 100}),
]


# ── Step 1: Generate all tensors (deterministic; safe to re-run) ──────────────
all_specs = []
for fm, _, _, _ in METHOD_SCHEDULE:
    for s in SEEDS:
        for r in RANKS:
            for snr in SNRS:
                spec = TensorSpec(dims=DIMS, rank=r, sparsity=SPARSITY,
                                  snr_db=snr, seed=s, factor_mode=fm)
                all_specs.append(spec)

generated = 0
for spec in all_specs:
    if not os.path.exists(f'{spec.cell_id()}.npy'):
        T, truth = generate_cp_tensor(spec)
        save_tensor(T, truth)
        generated += 1

print(f'Total specs: {len(all_specs)} | Newly generated: {generated} | Already on disk: {len(all_specs)-generated}')

In [ ]:
# ── Step 2: Run each method on its tensors ────────────────────────────────────
for fm, method, run_fn, kwargs in METHOD_SCHEDULE:

    if method != 'CP-ALS' and not MATLAB_AVAILABLE:
        print(f'Skipping {method} (MATLAB not available)')
        continue

    specs_for_method = [
        TensorSpec(dims=DIMS, rank=r, sparsity=SPARSITY, snr_db=snr, seed=s, factor_mode=fm)
        for s in SEEDS for r in RANKS for snr in SNRS
    ]

    print(f'\n── {method} ({len(specs_for_method)} runs) ──')
    for spec in specs_for_method:
        stem = spec.cell_id()
        # Skip if already logged
        if os.path.exists(RESULTS_CSV):
            existing = pd.read_csv(RESULTS_CSV)
            if ((existing['cell_id'] == stem) & (existing['method'] == method)).any():
                continue
        try:
            w, F, rt, extras = run_fn(stem, rank=spec.rank, **kwargs)
            rec = evaluate(stem, w, F, rt, method_name=method, extra=extras)
            log_result(rec)
            print(f'  seed={spec.seed:4d}  rank={spec.rank:3d}  snr={spec.snr_db:5}  '
                  f'ferr={rec["factor_err"]:.4f}  rerr={rec["recon_err"]:.4f}  t={rt:.1f}s')
        except Exception as e:
            print(f'  ERROR seed={spec.seed} rank={spec.rank}: {e}')

print('\nGrid sweep complete.')
df = pd.read_csv(RESULTS_CSV)
print(df.groupby('method')[['factor_err','recon_err','runtime_s']].mean().round(4))

## 9. Initialisation Sensitivity Sub-Experiment

Fix the tensor seed; vary `init_seed` across multiple random starts.  
Reveals algorithmic stability independent of data variance.

In [ ]:
INIT_SEEDS  = list(range(10))
FIXED_SPEC  = TensorSpec(dims=DIMS, rank=10, sparsity=SPARSITY, snr_db=20.0, seed=42,
                          factor_mode='symmetric_sparse')

T, truth = generate_cp_tensor(FIXED_SPEC)
save_tensor(T, truth)
stem = FIXED_SPEC.cell_id()

print('CP-ALS initialisation sensitivity:')
for i_seed in INIT_SEEDS:
    w, F, rt, extras = run_cp_als(stem, rank=FIXED_SPEC.rank, n_iter_max=300, init_seed=i_seed)
    rec = evaluate(stem, w, F, rt, method_name=f'CP-ALS-init{i_seed}',
                   extra={'init_seed': i_seed, 'experiment': 'init_sensitivity'})
    log_result(rec)
    print(f'  init_seed={i_seed}  factor_err={rec["factor_err"]:.4f}  recon_err={rec["recon_err"]:.4f}')

## 10. Plots

### 10a. Factor recovery error vs. SNR (all methods)

In [ ]:
df = pd.read_csv(RESULTS_CSV)

# Keep only primary methods (not init-sensitivity sub-experiment)
df_main = df[df['method'].isin(['CP-ALS', 'TensorNOODL', 'M2PALS'])].copy()
df_main['snr_db'] = pd.to_numeric(df_main['snr_db'], errors='coerce').fillna(np.inf)

agg = (
    df_main
    .groupby(['method', 'rank_true', 'snr_db'])
    .agg(ferr_mean=('factor_err', 'mean'), ferr_std=('factor_err', 'std'),
         rerr_mean=('recon_err',  'mean'), n=('factor_err', 'size'))
    .reset_index()
)

methods  = agg['method'].unique()
ranks    = sorted(agg['rank_true'].unique())
colors   = {'CP-ALS': '#2563EB', 'TensorNOODL': '#16A34A', 'M2PALS': '#DC2626'}
markers  = {'CP-ALS': 'o', 'TensorNOODL': 's', 'M2PALS': '^'}

fig, axes = plt.subplots(1, len(ranks), figsize=(5 * len(ranks), 4), sharey=True)
if len(ranks) == 1:
    axes = [axes]

for ax, rank in zip(axes, ranks):
    for method in methods:
        sub = agg[(agg['method'] == method) & (agg['rank_true'] == rank)].sort_values('snr_db')
        if sub.empty:
            continue
        snr_vals = sub['snr_db'].replace(np.inf, 40.0)  # plot inf as 40
        ax.errorbar(snr_vals, sub['ferr_mean'], yerr=sub['ferr_std'],
                    marker=markers[method], color=colors[method],
                    label=method, capsize=4, linewidth=1.8)
    ax.set_title(f'Rank = {rank}', fontsize=11)
    ax.set_xlabel('SNR (dB)  [40 = noiseless]')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Factor recovery error (mean ± std)')
fig.suptitle('CP Decomposition Benchmark — Factor Recovery by Method', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('benchmark_factor_err.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── 10b. Runtime vs rank ──────────────────────────────────────────────────────
rt_agg = (
    df_main
    .groupby(['method', 'rank_true'])
    .agg(rt_mean=('runtime_s', 'mean'), rt_std=('runtime_s', 'std'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(6, 4))
for method in methods:
    sub = rt_agg[rt_agg['method'] == method].sort_values('rank_true')
    if sub.empty:
        continue
    ax.errorbar(sub['rank_true'], sub['rt_mean'], yerr=sub['rt_std'],
                marker=markers[method], color=colors[method],
                label=method, capsize=4, linewidth=1.8)

ax.set_xlabel('True CP Rank')
ax.set_ylabel('Runtime (s)')
ax.set_title('Runtime vs. Rank')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('benchmark_runtime.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── 10c. M2PALS atom-selection accuracy ───────────────────────────────────────
df_m2pals = df_main[df_main['method'] == 'M2PALS'].copy()

if 'atom_selection_acc' in df_m2pals.columns and not df_m2pals.empty:
    asa_agg = (
        df_m2pals
        .groupby(['rank_true', 'snr_db'])
        .agg(asa_mean=('atom_selection_acc', 'mean'),
             asa_std=('atom_selection_acc',  'std'))
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(6, 4))
    for rank in sorted(asa_agg['rank_true'].unique()):
        sub = asa_agg[asa_agg['rank_true'] == rank].sort_values('snr_db')
        snr_vals = sub['snr_db'].replace(np.inf, 40.0)
        ax.errorbar(snr_vals, sub['asa_mean'], yerr=sub['asa_std'],
                    marker='^', capsize=4, label=f'rank={rank}')
    ax.set_xlabel('SNR (dB)  [40 = noiseless]')
    ax.set_ylabel('Atom selection accuracy')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title('M2PALS — Atom Selection Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('benchmark_m2pals_asa.pdf', bbox_inches='tight')
    plt.show()
else:
    print('No M2PALS atom_selection_acc data yet.')

## 11. Next Steps

1. **Set env vars** before running:
   ```bash
   export TENSOR_NOODL_DIR=/path/to/TensorNOODL
   export M2PALS_DIR=/path/to/Tensor_codes/Dictionary-based_decomposition/M2PALS
   ```
2. **Install MATLAB engine**: `pip install matlabengine` (requires MATLAB R2021b+).
3. **Tune TensorNOODL step sizes** — edit `_NOODL_ETA_TABLE` or pass `eta_A` explicitly per rank.
4. **Add the original M2PALS MATLAB code** — the bridge script `run_m2pals_bridge.m` written above is a faithful reimplementation of the ALS core. If you have the original `M2PALS.m`, replace the bridge body with a call to it: `[A,B,C,lambda] = M2PALS(T, {D}, [rank, rank], n_iter);`
5. **Scale the grid** — uncomment the full-grid `SEEDS`/`RANKS`/`SNRS` block in §8.
6. **Real-data section** — write `load_real_tensor(name)` that bypasses generation; the `run_<method>` interface stays identical.
7. **Reproducibility audit** — re-run on a clean machine; `runs_v2.csv` should be bitwise-identical.